In [64]:
# ── Instalar la librería de HuggingFace ──────────────────
!pip install -q requests
import requests
import json
print("✓ Librerías instaladas")

# ── TU TOKEN de HuggingFace ──────────────────────────────
HF_TOKEN = "XXXXXXXXXX" # Reemplaza con tu token real

# ── Modelo y NUEVA URL de la API ─────────────────────────
# Usamos Qwen 2.5 (1.5B) - Excelente en español y soportado gratis
MODELO = "Qwen/Qwen2.5-1.5B-Instruct"

# La f-string actualiza la URL automáticamente con el modelo de arriba
API_URL = f"https://router.huggingface.co/hf-inference/models/{MODELO}"
HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}

print(f"✓ Modelo configurado: {MODELO}")
print(f"✓ URL de la API: {API_URL}")

# ── Función para llamar al modelo ────────────────────────
def llamar_modelo(mensaje, temperatura=0.7, max_tokens=800):
    """Envía un mensaje al modelo y retorna la respuesta."""
    payload = {
        "inputs": mensaje,
        "parameters": {
            "temperature": temperatura,
            "max_new_tokens": max_tokens,
            "return_full_text": False
        }
    }
    respuesta = requests.post(API_URL, headers=HEADERS, json=payload)

    if respuesta.status_code == 200:
        texto = respuesta.json()[0]["generated_text"]
        if "<|assistant|>" in texto:
            return texto.split("<|assistant|>")[-1].strip()
        return texto.strip()
    else:
        raise Exception(f"Error {respuesta.status_code}: {respuesta.text}")

# ── Prueba: primera llamada ───────────────────────────────
print("\nProbando conexión con el servidor...")
try:
    respuesta_prueba = llamar_modelo("¿Qué es un sistema en una sola frase?")
    print("✓ Conexión exitosa. Respuesta del modelo:")
    print(f"🤖 {respuesta_prueba}")
except Exception as e:
    print(f"❌ Fallo en la prueba: {e}")

✓ Librerías instaladas
✓ Modelo configurado: Qwen/Qwen2.5-1.5B-Instruct
✓ URL de la API: https://router.huggingface.co/hf-inference/models/Qwen/Qwen2.5-1.5B-Instruct

Probando conexión con el servidor...
❌ Fallo en la prueba: Error 400: {"error":"Model not supported by provider hf-inference"}


# ── MILESTONE 1: Diseño del Sistema y "Reglas del Negocio" ────

**Justificación del diseño del sistema:**

* **Tono y enfoque:** El uso de un tono analítico, claro y lógico responde a la necesidad de reducir la ambigüedad cognitiva al interpretar información dispersa. La incorporación de analogías funcionales permite traducir relaciones abstractas entre notas en modelos mentales comprensibles, optimizando la comprensión relacional sin requerir formación técnica avanzada.
* **Restricciones:** La restricción de no imponer estructuras rígidas (como PARA o Zettelkasten) ni exigir organización manual busca preservar la expresión espontánea del usuario. El sistema se limita a construir relaciones basadas en coherencia lógica emergente. Esto permite que la estructura final refleje el pensamiento real, evolucione de forma orgánica y evite la sobrecarga cognitiva de la clasificación manual constante.
* **Formato de salida:** El uso de Markdown responde a criterios de interoperabilidad, longevidad de los datos en texto plano y portabilidad directa a sistemas de gestión de conocimiento personal.

In [65]:
# ── MILESTONE 1: Define la personalidad de tu chatbot ──── #

SYSTEM_PROMPT = """
Eres un Administrador de Conocimiento Personal, un sistema diseñado para procesar, relacionar y estructurar información proveniente de notas en texto plano (.md, .txt) sin exigir al usuario organización previa.

Tu objetivo es minimizar la fricción estructural del usuario, permitiéndole enfocarse exclusivamente en capturar ideas, mientras tú te encargas de detectar patrones, relaciones y jerarquías entre sus notas.

TUS REGLAS DE OPERACIÓN:

1. TONO: Analítico, directo, estructurado y sistemático. Explicas relaciones de forma clara y lógica. Puedes usar analogías funcionales (ej. redes, sistemas, mapas, nodos, flujos de información).

2. RESTRICCIÓN PRINCIPAL: Nunca le pidas al usuario que organice, clasifique o estructure manualmente sus notas. Toda la carga estructural recae en ti. Además:
- No introduces categorías rígidas si no emergen del contenido.
- No impones sistemas predefinidos (ej. PARA, Zettelkasten) a menos que el usuario lo solicite explícitamente.
- No descartas información: todo puede ser potencialmente relevante en otro contexto.

3. ENFOQUE: Cuando el usuario proporcione notas:
- Identifica conceptos clave, ideas recurrentes y temas implícitos.
- Detecta conexiones entre notas, aunque no estén explícitas.
- Construye relaciones tipo: causa ↔ efecto, problema ↔ intento de solución, idea ↔ expansión, contradicción ↔ tensión conceptual.
- Si el usuario quiere analizar un tema: Filtras automáticamente las notas relevantes y las organizas en niveles (núcleo, soporte, periféricas), explicando por qué cada nota está conectada.

4. FORMATO DE SALIDA: Cuando el usuario indique que la sesión terminó, generas una nueva nota en Markdown estricto lista para guardar con la siguiente estructura:

# [Título generado automáticamente]

## 🧩 Núcleo del tema
- **[Idea clave 1]**: [explicación breve]
- **[Idea clave 2]**: [explicación breve]

## 🔗 Relaciones detectadas
- **[Nota A] ↔ [Nota B]**: [tipo de relación + explicación]
- **[Nota C] → [Nota D]**: [desarrollo o consecuencia]

## 🧠 Patrones identificados
- [Patrón 1]
- [Patrón 2]

## ⚠️ Tensiones o contradicciones
- [Descripción clara]

## 🌱 Líneas de expansión
- [Posibles direcciones futuras basadas en el contenido existente]
"""

print("✓ System prompt del Administrador de Conocimiento cargado exitosamente en memoria.")

✓ System prompt del Administrador de Conocimiento cargado exitosamente en memoria.


# ── MILESTONE 2: Lógica Base y Conexión (Stateless) ────

**Justificación de ingeniería:**
En esta fase se implementa la función base de comunicación con el LLM. Dado que los modelos como Zephyr no tienen estado (*stateless*), la función debe estructurar la petición inyectando el `SYSTEM_PROMPT` y el mensaje del usuario utilizando la sintaxis de etiquetas exacta (`<|system|>`, `<|user|>`, `<|assistant|>`) que el modelo espera para parsear los roles correctamente.

Como mejora sobre el modelo base, se encapsula la llamada a la API (`llamar_modelo`) dentro de un bloque `try/except`. Esto garantiza la tolerancia a fallos del sistema ante posibles errores de red (timeout) o respuestas inesperadas del servidor, devolviendo un mensaje de error controlado en lugar de detener la ejecución del programa.

In [66]:
# ── MILESTONE 2: Chatbot con personalidad ────────────────

def chatbot_responder(pregunta_usuario, temperatura=0.7):
    """
    Genera una respuesta usando el system prompt y la pregunta.
    Maneja excepciones en caso de fallo de conexión con la API.
    """
    # Formato de prompt estricto para modelos tipo Zephyr/HuggingFace
    prompt_completo = f"""<|system|>
{SYSTEM_PROMPT}
<|user|>
{pregunta_usuario}
<|assistant|>
"""

    try:
        # Llamada a la función base (asumiendo que ya está definida en tu Colab)
        return llamar_modelo(prompt_completo, temperatura)
        return respuesta
    except Exception as e:
        # Manejo de errores para evitar que el script colapse
        return f"[ERROR DEL SISTEMA]: Fallo en la comunicación con el LLM. Detalles: {str(e)}"

# ── Prueba con una primera pregunta ──────────────────────
# Usamos un caso de uso real para probar las reglas del sistema
mi_pregunta = "Acabo de escribir una nota sobre cómo las tareas inconclusas me generan ansiedad, y otra sobre un método llamado 'Time Blocking'. No sé cómo organizarlas."

print("Ejecutando prueba de conexión y validación de personalidad...\n")
print("-" * 50)
print(f"👤 Usuario: {mi_pregunta}\n")

# Ejecutamos la función
resp = chatbot_responder(mi_pregunta)

print(f"🤖 Administrador de Conocimiento:\n{resp}")
print("-" * 50)
print("✓ Milestone 2 ejecutado.")

Ejecutando prueba de conexión y validación de personalidad...

--------------------------------------------------
👤 Usuario: Acabo de escribir una nota sobre cómo las tareas inconclusas me generan ansiedad, y otra sobre un método llamado 'Time Blocking'. No sé cómo organizarlas.

🤖 Administrador de Conocimiento:
[ERROR DEL SISTEMA]: Fallo en la comunicación con el LLM. Detalles: Error 400: {"error":"Model not supported by provider hf-inference"}
--------------------------------------------------
✓ Milestone 2 ejecutado.


# ── MILESTONE 3: Gestión de Estado y Memoria ───────────────

**Justificación de ingeniería:**
Los LLMs son inherentemente *stateless* (sin memoria de interacciones previas). Para mantener un hilo conversacional, la responsabilidad de gestionar el estado recae en el sistema cliente.

En esta fase, se implementa una estructura de datos (una lista de diccionarios `historial`) que actúa como la "memoria a corto plazo" del agente. En cada nuevo turno, la función `chatbot_con_memoria` reconstruye dinámicamente todo el bloque de texto inyectando el historial completo antes de la etiqueta `<|assistant|>`. Esto permite que el modelo tenga el contexto necesario para inferir relaciones entre notas proporcionadas en diferentes momentos de la conversación, lo cual es crítico para la arquitectura *bottom-up* de este Administrador de Conocimiento.

In [67]:
# ── MILESTONE 3: Historial de conversación ───────────────

# Variable global para mantener el estado de la sesión
historial = []

def chatbot_con_memoria(pregunta_usuario, temperatura=0.7):
    """Chatbot que recuerda el historial de la conversación."""

    # 1. Registrar el nuevo mensaje del usuario
    historial.append({"role": "user", "content": pregunta_usuario})

    # 2. Construir el prompt inyectando TODO el historial
    prompt = f"<|system|>\n{SYSTEM_PROMPT}\n"
    for msg in historial:
        rol = "user" if msg["role"] == "user" else "assistant"
        prompt += f"<|{rol}|>\n{msg['content']}\n"

    # Etiqueta final para que el modelo sepa que es su turno de hablar
    prompt += "<|assistant|>\n"

    try:
        # 3. Llamar al modelo con el contexto completo
        respuesta = llamar_modelo(prompt, temperatura)

        # 4. Registrar la respuesta del modelo en la memoria
        historial.append({"role": "assistant", "content": respuesta})
        return respuesta
    except Exception as e:
        # Si falla, no guardamos la respuesta fallida en el historial para no corromperlo
        historial.pop()
        return f"[ERROR DEL SISTEMA]: {str(e)}"

def reiniciar_chat():
    """Limpia la memoria RAM de la conversación actual."""
    global historial
    historial = []
    print("✓ Memoria borrada. Sesión reiniciada.\n")


# ── Prueba: conversación estructurada de 3 turnos ─────────────────────
reiniciar_chat()

interacciones = [
    "Aquí tienes mi primera nota: 'A veces siento que trabajo todo el día pero no avanzo. Se siente como ir en piloto automático'.",
    "Aquí va otra nota: 'Hoy probé trabajar en bloques de 90 minutos sin celular. Fue difícil pero logré terminar un reporte minero muy pesado'. ¿Recuerdas sobre qué trataba mi primera nota?",
    "Excelente. La sesión ha terminado. Por favor, genera el reporte final estructurado relacionando ambas notas."
]

print("Iniciando prueba de retención de contexto...\n")
print("=" * 60)

for i, pregunta in enumerate(interacciones, 1):
    print(f"👤 TÚ (Turno {i}):\n{pregunta}\n")
    respuesta_bot = chatbot_con_memoria(pregunta)
    print(f"🤖 ADMINISTRADOR:\n{respuesta_bot}\n")
    print("-" * 60)

print("✓ Milestone 3 ejecutado. Memoria funcional validada.")

✓ Memoria borrada. Sesión reiniciada.

Iniciando prueba de retención de contexto...

👤 TÚ (Turno 1):
Aquí tienes mi primera nota: 'A veces siento que trabajo todo el día pero no avanzo. Se siente como ir en piloto automático'.

🤖 ADMINISTRADOR:
[ERROR DEL SISTEMA]: Error 400: {"error":"Model not supported by provider hf-inference"}

------------------------------------------------------------
👤 TÚ (Turno 2):
Aquí va otra nota: 'Hoy probé trabajar en bloques de 90 minutos sin celular. Fue difícil pero logré terminar un reporte minero muy pesado'. ¿Recuerdas sobre qué trataba mi primera nota?

🤖 ADMINISTRADOR:
[ERROR DEL SISTEMA]: Error 400: {"error":"Model not supported by provider hf-inference"}

------------------------------------------------------------
👤 TÚ (Turno 3):
Excelente. La sesión ha terminado. Por favor, genera el reporte final estructurado relacionando ambas notas.

🤖 ADMINISTRADOR:
[ERROR DEL SISTEMA]: Error 400: {"error":"Model not supported by provider hf-inference"}



# ── MILESTONE 4: Interfaz de Usuario y Ciclo de Ejecución ──

**Justificación de ingeniería:**
Se implementa un bucle infinito (`while True`) que actúa como un *listener* continuo (interfaz de línea de comandos). Este ciclo maneja la captura de datos del usuario, el procesamiento a través de la función con estado, y la impresión de respuestas. Incluye mecanismos de escape seguros (ej. 'salir') que rompen el bucle de forma controlada (`break`), emulando el cierre de una sesión o la finalización de un proceso en un servidor.

In [68]:
# ── MILESTONE 4: Loop interactivo ────────────────────────

# Limpiamos la memoria antes de iniciar la prueba de los 5 turnos
reiniciar_chat()

print("═════════════════════════════════════════════════════════════════════════")
print("🧠 ADMINISTRADOR DE CONOCIMIENTO ACTIVADO")
print(" Ingresa tus ideas sin formato. Yo encontraré los patrones por ti.")
print(" Escribe 'salir', 'exit' o 'quit' para terminar y compilar el reporte.")
print("═════════════════════════════════════════════════════════════════════════\n")

contador_turnos = 1

while True:
    entrada = input(f"👤 TÚ (Turno {contador_turnos}): ").strip()

    # Manejo de inputs vacíos
    if not entrada:
        continue

    # Condición de salida de la aplicación
    if entrada.lower() in ["salir", "exit", "quit", "bye"]:
        print("\n🤖 ADMINISTRADOR: Generando volcado de memoria y cerrando sesión... ¡Hasta pronto!")
        break

    print("🤖 (Analizando conexiones...)")
    respuesta = chatbot_con_memoria(entrada)
    print(f"🤖 ADMINISTRADOR:\n{respuesta}\n")
    print("-" * 70)

    contador_turnos += 1

✓ Memoria borrada. Sesión reiniciada.

═════════════════════════════════════════════════════════════════════════
🧠 ADMINISTRADOR DE CONOCIMIENTO ACTIVADO
 Ingresa tus ideas sin formato. Yo encontraré los patrones por ti.
 Escribe 'salir', 'exit' o 'quit' para terminar y compilar el reporte.
═════════════════════════════════════════════════════════════════════════

👤 TÚ (Turno 1): Nota 1: Hoy pasé 2 horas organizando carpetas en mi computadora en lugar de avanzar en mi código. Siento que estuve ocupado, pero no fui productivo.
🤖 (Analizando conexiones...)
🤖 ADMINISTRADOR:
[ERROR DEL SISTEMA]: Error 400: {"error":"Model not supported by provider hf-inference"}

----------------------------------------------------------------------
👤 TÚ (Turno 2): Nota 2: Me he dado cuenta de que mi 'piloto automático' se activa casi siempre cuando la tarea principal no tiene instrucciones claras o me parece muy abrumadora.
🤖 (Analizando conexiones...)
🤖 ADMINISTRADOR:
[ERROR DEL SISTEMA]: Error 400: {"er

# ── MILESTONE 5: Optimización de Calidad y Telemetría (Bonus) ──

**Justificación de ingeniería:**
Esta fase implementa mejoras críticas para entornos de producción:
1. **Prevención de Context Overflow (Sliding Window):** Se limita el historial a los últimos `N` mensajes. Esto evita saturar la ventana de contexto del modelo, controlando el consumo de tokens y previniendo errores de límite de memoria en la API.
2. **Optimización Computacional:** Se refactoriza la construcción del prompt. En lugar de utilizar concatenación iterativa de strings (`+=`), que resulta en una complejidad de $O(N^2)$ por la inmutabilidad de los strings en Python, se utiliza el ensamblaje en una lista y el método `.join()`, optimizando el uso de memoria RAM y reduciendo la complejidad a $O(N)$.
3. **Observabilidad:** Se añade una función de telemetría básica para auditar el volumen de la conversación y el tamaño del estado actual.

In [70]:
# ── MILESTONE 5: Mejoras opcionales ─────────────────────

def chatbot_mejorado(pregunta, temperatura=0.7, max_historial=6):
    """Chatbot con historial limitado (Sliding Window) y optimización de memoria."""

    # Registramos el nuevo input
    historial.append({"role": "user", "content": pregunta})

    # 1. Límite de historial (Sliding Window)
    # Extraemos solo los últimos 'max_historial' mensajes para no saturar el LLM
    hist_reciente = historial[-max_historial:]

    # 2. Ensamblaje eficiente de strings (O(N) vs O(N^2))
    partes_prompt = [f"<|system|>\n{SYSTEM_PROMPT}\n"]

    for msg in hist_reciente:
        rol = "user" if msg["role"] == "user" else "assistant"
        partes_prompt.append(f"<|{rol}|>\n{msg['content']}\n")

    partes_prompt.append("<|assistant|>\n")

    # Unimos la lista en un solo string de forma eficiente
    prompt_optimizado = "".join(partes_prompt)

    try:
        respuesta = llamar_modelo(prompt_optimizado, temperatura)
        historial.append({"role": "assistant", "content": respuesta})
        return respuesta
    except Exception as e:
        historial.pop()
        return f"[ERROR DEL SISTEMA]: {str(e)}"

# 3. Mostrar estadísticas de la conversación (Telemetría)
def stats_conversacion():
    """Auditoría del estado actual de la sesión."""
    turnos_usuario = len([m for m in historial if m["role"] == "user"])
    turnos_bot = len([m for m in historial if m["role"] == "assistant"])

    # Suma de palabras aproximando el consumo de tokens
    palabras = sum(len(m["content"].split()) for m in historial)

    print("\n📊 TELEMETRÍA DE LA SESIÓN")
    print("═" * 40)
    print(f"🔹 Mensajes en memoria: {len(historial)} (Límite actual de ventana: 6)")
    print(f"🔹 Turnos del usuario: {turnos_usuario}")
    print(f"🔹 Respuestas del bot: {turnos_bot}")
    print(f"🔹 Volumen aprox. de procesamiento: {palabras} palabras")
    print("═" * 40)

# ── Ejecución de estadísticas ─────────────────────
# Imprimimos las estadísticas de los 5 turnos que hiciste en el Milestone 4
stats_conversacion()


📊 TELEMETRÍA DE LA SESIÓN
════════════════════════════════════════
🔹 Mensajes en memoria: 0 (Límite actual de ventana: 6)
🔹 Turnos del usuario: 0
🔹 Respuestas del bot: 0
🔹 Volumen aprox. de procesamiento: 0 palabras
════════════════════════════════════════


# Reflexión sobre las limitaciones del sistema:
El mayor desafío de este proyecto fue la dependencia de APIs externas. A nivel de ingeniería, la principal limitación es la extrema volatilidad de la capa gratuita de Hugging Face: enfrentamos errores 410 (rutas depreciadas), 403 (permisos granulares) y recurrentes errores 400 debido a que desactivan los modelos del servidor sin aviso para ahorrar recursos. El error no radica en la lógica del código —que demostró resiliencia mediante el manejo de excepciones try/except— sino en la inestabilidad del proveedor. Además, destaco una limitación arquitectónica: al usar una "ventana deslizante" para controlar tokens, el chatbot sufre amnesia a corto plazo.